# Dashboard Data Prep & Chart Prototyping
### Data Viz Group Project — Streamlit Dashboard Skeleton

This notebook is a **prototyping space**, not the deployed app. Streamlit apps run as a
`.py` script (`app.py`), so once a chart looks right here, copy the finalized encoding
logic into `app.py`.

Use this notebook to:
- Load and clean `merged_env_health_2023.csv` the same way `app.py` does
- Sanity-check the variable groups (SES / AQI / Emissions / Health)
- Prototype and preview the linked scatter + bar brushing chart before it goes in the app


In [1]:
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

## 1. Load and clean the merged dataset
(Same cleaning steps as `app.py`, kept in sync so both stay consistent.)

In [2]:
df = pd.read_csv("merged_env_health_2023.csv")

# TotalPopulation arrives as a comma-formatted string ("24,434")
df["TotalPopulation"] = pd.to_numeric(
    df["TotalPopulation"].astype(str).str.replace(",", ""), errors="coerce"
)

numeric_cols = [
    "total_emissions", "co2_emissions", "ch4_emissions", "n2o_emissions",
    "num_facilities", "Median AQI", "Max AQI", "90th Percentile AQI",
    "Days with AQI", "Good Days", "Moderate Days", "Unhealthy Days",
    "Very Unhealthy Days", "Hazardous Days", "Unhealthy for Sensitive Groups Days",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["emissions_per_capita"] = df["total_emissions"] / df["TotalPopulation"]
df["log_total_emissions"] = np.log1p(df["total_emissions"])
df["log_emissions_per_capita"] = np.log1p(df["emissions_per_capita"])

df.shape


(3192, 72)

## 2. Variable groups
Mirrors the `SES` / `AQI` / `EMISSIONS` / `HEALTH` groups from the EDA notebook — these power the dropdowns in `app.py`.

In [3]:
VARIABLE_GROUPS = {
    "Socioeconomic (SES)": {
        "Food insecurity (%)": "FOODINSECU",
        "Housing insecurity (%)": "HOUSINSECU",
        "Food stamp / SNAP use (%)": "FOODSTAMP",
        "Lack of health insurance (%)": "ACCESS2",
        "Utility shutoff threat (%)": "SHUTUTILITY",
        "Lack of transportation (%)": "LACKTRPT",
        "Lack of social/emotional support (%)": "EMOTIONSPT",
    },
    "Air Quality (AQI)": {
        "Median AQI": "Median AQI",
        "Max AQI": "Max AQI",
        "Days with AQI reported": "Days with AQI",
        "Unhealthy days": "Unhealthy Days",
        "PM2.5 days": "Days PM2.5",
    },
    "GHG Emissions": {
        "Total emissions (log)": "log_total_emissions",
        "Emissions per capita (log)": "log_emissions_per_capita",
        "Number of reporting facilities": "num_facilities",
    },
    "Health Outcomes (PLACES)": {
        "Mental health, not good days (%)": "MHLTH",
        "Asthma (%)": "CASTHMA",
        "Obesity (%)": "OBESITY",
        "Diabetes (%)": "DIABETES",
    },
}

for group, mapping in VARIABLE_GROUPS.items():
    cols = list(mapping.values())
    pct_missing = df[cols].isna().mean().mean() * 100
    print(f"{group:28s} avg missing: {pct_missing:5.1f}%")


Socioeconomic (SES)          avg missing:  25.0%
Air Quality (AQI)            avg missing:  61.1%
GHG Emissions                avg missing:  40.5%
Health Outcomes (PLACES)     avg missing:   7.4%


## 3. Prototype: linked scatter (brush) + coordinated bar chart
Pick any X/Y pair below, run the cell, and drag a box on the scatter to see the bar chart update — this is the exact pattern used in `app.py`.

In [4]:
x_col = "log_total_emissions"
y_col = "MHLTH"

plot_df = df.dropna(subset=[x_col, y_col, "state_abbr"]).copy()

brush = alt.selection_interval(encodings=["x", "y"])

scatter = (
    alt.Chart(plot_df)
    .mark_circle(size=65, opacity=0.65)
    .encode(
        x=alt.X(f"{x_col}:Q", scale=alt.Scale(zero=False)),
        y=alt.Y(f"{y_col}:Q", scale=alt.Scale(zero=False)),
        color=alt.condition(brush, alt.value("#1C7293"), alt.value("#D9D9D9")),
        tooltip=["county_name:N", "state_abbr:N", f"{x_col}:Q", f"{y_col}:Q"],
    )
    .add_params(brush)
    .properties(width=420, height=400, title=f"{x_col} vs {y_col}")
)

bar = (
    alt.Chart(plot_df)
    .mark_bar(color="#065A82")
    .encode(
        y=alt.Y("state_abbr:N", sort="-x", title="State"),
        x=alt.X(f"mean({y_col}):Q", title=f"Mean {y_col}"),
        tooltip=["state_abbr:N", f"mean({y_col}):Q", "count():Q"],
    )
    .transform_filter(brush)
    .properties(width=320, height=400, title="Mean by state (brushed only)")
)

alt.hconcat(scatter, bar)


alt.HConcatChart(...)

## 4. Next steps
- Swap in different `x_col` / `y_col` pairs above to sanity-check each variable group renders sensibly (watch for high missingness — AQI is only available for ~500 counties).
- Once the team agrees on labels/colors, those choices are already wired into `app.py`'s dropdowns — no need to duplicate logic, just update `VARIABLE_GROUPS` in both places if you add variables.
- Screenshots of the deployed dashboard (from Streamlit Community Cloud) go into the PPTX report slides.
